In [1]:
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("postgresql+psycopg2://postgres@127.0.0.1:5432/trading")

def load_bars(tf):
    b = pd.read_sql(
        "SELECT ts_utc, open, high, low, close, atr_14, session FROM fx_bars "
        "WHERE pair = 'EURUSD' AND timeframe = %(tf)s ORDER BY ts_utc",
        engine, params={"tf": tf}, index_col="ts_utc",
    )
    b.index = pd.to_datetime(b.index, utc=True)
    return b

b15 = load_bars("15m")
b60 = load_bars("1h")
b15 = b15[b15.index < "2024-01-01"]     # BUILD + VALIDATION (2024-25 test stays locked)
b60 = b60[b60.index < "2024-01-01"]
print(len(b15), len(b60))

213212 53319


In [2]:
import numpy as np
import pandas as pd

PIP = 0.0001
COST_PIPS = 1.4          # spread + slippage, round trip (from the specs)

def simulate_trade(bars, entry_i, side, stop, target, max_bars=None, exit_time=None):
    """
    bars:      DataFrame with open/high/low/close, index = bar start time (UTC)
    entry_i:   integer position of the bar we ENTER on (fill at its open)
    side:      +1 long, -1 short
    stop, target: absolute prices
    max_bars:  close at that bar's close if still open after this many bars
    exit_time: 'HH:MM' UTC, close at the open of the first bar at/after this time
    Returns a dict describing the trade, with P&L in pips after costs.
    """
    entry_px = bars["open"].iat[entry_i]
    n = len(bars)
    for k, i in enumerate(range(entry_i, n)):
        t = bars.index[i]
        o, h, l, c = bars["open"].iat[i], bars["high"].iat[i], bars["low"].iat[i], bars["close"].iat[i]

        # time exit: leave at this bar's open
        if exit_time is not None and k > 0 and t.strftime("%H:%M") >= exit_time:
            exit_px, reason = o, "time"
            break

        if side == 1:
            hit_stop, hit_tgt = l <= stop, h >= target
        else:
            hit_stop, hit_tgt = h >= stop, l <= target

        if hit_stop:                       # checked FIRST: worst case if both touched
            # if the bar opened beyond the stop (a gap), we fill at the open, not the stop
            gap = (o < stop) if side == 1 else (o > stop)
            exit_px, reason = (o if gap else stop), "stop"
            break
        if hit_tgt:
            exit_px, reason = target, "target"
            break
        if max_bars is not None and k + 1 >= max_bars:
            exit_px, reason = c, "max_bars"
            break
    else:
        exit_px, reason, i = bars["close"].iat[n - 1], "end_of_data", n - 1

    gross = side * (exit_px - entry_px) / PIP
    return {
        "entry_time": bars.index[entry_i], "exit_time": bars.index[i],
        "side": side, "entry": entry_px, "exit": exit_px, "reason": reason,
        "bars_held": i - entry_i + 1,
        "pips_gross": gross, "pips_net": gross - COST_PIPS,
        "risk_pips": abs(entry_px - stop) / PIP,
    }

In [3]:
def run_strategy_b(b60, band=2.0, stop_mult=1.5, max_bars=8):
    trades, skipped = [], {"no_data": 0, "high_vol": 0, "target_wrong_side": 0}
    upper = b60["ma20"] + band * b60["sd20"]
    lower = b60["ma20"] - band * b60["sd20"]
    i, n = 0, len(b60)
    while i < n - 1:
        t = b60.index[i]
        c = b60["close"].iat[i]
        side = 1 if c < lower.iat[i] else (-1 if c > upper.iat[i] else 0)
        if side == 0 or t.hour >= 7:                  # signal bar must START in 00:00-06:00 UTC
            i += 1; continue
        atr, cap, ma = b60["atr_14"].iat[i], b60["atr_cap"].iat[i], b60["ma20"].iat[i]
        if np.isnan(atr) or np.isnan(cap) or np.isnan(ma):
            skipped["no_data"] += 1; i += 1; continue
        if atr > cap:
            skipped["high_vol"] += 1; i += 1; continue
        entry_i = i + 1                                # fill at the NEXT bar's open
        entry_px = b60["open"].iat[entry_i]
        target = ma                                    # fixed at signal time
        if side * (target - entry_px) <= 0:            # already snapped back by the open
            skipped["target_wrong_side"] += 1; i += 1; continue
        stop = entry_px - side * stop_mult * atr
        tr = simulate_trade(b60, entry_i, side, stop, target, max_bars=max_bars)
        trades.append(tr)
        i = b60.index.get_loc(tr["exit_time"]) + 1     # one position at a time
    return pd.DataFrame(trades), skipped


In [4]:
b60 = b60.copy()
b60["ma20"] = b60["close"].rolling(20).mean()
b60["sd20"] = b60["close"].rolling(20).std()
b60["atr_cap"] = b60["atr_14"].shift(1).rolling(60 * 24, min_periods=500).quantile(0.80)

# features, all computed from the signal bar or earlier
f = pd.DataFrame(index=b60.index)
f["z"]        = (b60["close"] - b60["ma20"]) / b60["sd20"]                   # how stretched
f["vol_rank"] = b60["atr_14"] / b60["atr_14"].shift(1).rolling(60 * 24, min_periods=500).median()
f["trend_24"] = (b60["close"] - b60["close"].shift(24)) / b60["atr_14"]      # 24h move in ATRs
f["band_w"]   = (4 * b60["sd20"]) / b60["atr_14"]                            # band width in ATRs
day_open = b60["open"].groupby(b60.index.normalize()).transform("first")
f["from_open"] = (b60["close"] - day_open) / b60["atr_14"]
f["hour"]     = b60.index.hour
f["dow"]      = b60.index.dayofweek

tr, _ = run_strategy_b(b60, band=1.5, stop_mult=2.0)
sig_pos = [b60.index.get_loc(t) - 1 for t in tr["entry_time"]]               # signal bar = bar before entry
X = f.iloc[sig_pos].reset_index(drop=True)
X["side_z"] = X["z"] * tr["side"].values             # stretch in the trade's direction
X["side_trend"] = X["trend_24"] * tr["side"].values  # positive = trading WITH the 24h move
data = pd.concat([tr.reset_index(drop=True), X], axis=1)
data["R"] = data["pips_net"] / data["risk_pips"]
data["y"] = (data["pips_net"] > 0).astype(int)
data["year"] = data["entry_time"].dt.year
print(data.groupby("year").agg(trades=("R", "size"), win=("y", "mean"), avg_R=("R", "mean")).round(3))

      trades    win  avg_R
year                      
2015      73  0.521 -0.008
2016      75  0.480 -0.082
2017      80  0.550 -0.013
2018      72  0.556 -0.002
2019      60  0.533 -0.067
2020      84  0.536  0.002
2021      88  0.534 -0.060
2022      78  0.526 -0.043
2023      44  0.545 -0.028


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

feats = ["side_z", "vol_rank", "side_trend", "band_w", "from_open", "hour", "dow"]
models = {
    "logit": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "forest": lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42),
}
for name, make in models.items():
    data[f"p_{name}"] = np.nan
    for yr in range(2017, 2024):
        train = data[data.year < yr]
        test  = data.year == yr
        if test.sum() == 0:
            print(name, yr, "-> no trades, skipped")
            continue
        m = make().fit(train[feats], train["y"])
        data.loc[test, f"p_{name}"] = m.predict_proba(data.loc[test, feats])[:, 1]
print(data[["p_logit", "p_forest"]].describe().round(3))

       p_logit  p_forest
count  506.000   506.000
mean     0.518     0.523
std      0.094     0.067
min      0.160     0.360
25%      0.461     0.479
50%      0.526     0.514
75%      0.588     0.566
max      0.786     0.734


In [6]:
oos = data[data.year >= 2017]
build, valid = oos[oos.year <= 2021], oos[oos.year >= 2022]
for name in ["logit", "forest"]:
    print(f"\n== {name} ==   baseline (take all): build {build.R.mean():.3f} R | valid {valid.R.mean():.3f} R")
    for th in [0.45, 0.50, 0.55, 0.60]:
        b_keep, v_keep = build[build[f"p_{name}"] > th], valid[valid[f"p_{name}"] > th]
        print(f"threshold {th:.2f}: build {len(b_keep):4d} trades {b_keep.R.mean():.3f} R | "
              f"valid {len(v_keep):4d} trades {v_keep.R.mean():.3f} R")


== logit ==   baseline (take all): build -0.027 R | valid -0.038 R
threshold 0.45: build  292 trades -0.039 R | valid  108 trades -0.065 R
threshold 0.50: build  233 trades -0.015 R | valid   84 trades -0.057 R
threshold 0.55: build  148 trades -0.007 R | valid   51 trades 0.080 R
threshold 0.60: build   83 trades -0.020 R | valid   24 trades 0.064 R

== forest ==   baseline (take all): build -0.027 R | valid -0.038 R
threshold 0.45: build  331 trades -0.063 R | valid  107 trades -0.035 R
threshold 0.50: build  214 trades -0.095 R | valid   83 trades -0.037 R
threshold 0.55: build  107 trades -0.026 R | valid   55 trades 0.007 R
threshold 0.60: build   48 trades -0.126 R | valid   18 trades -0.191 R
